# 04 — DYNAMIC_FILTERED_MAS_RAG

```
User Question
     │
     ▼
┌─ FFN GATING ───────────────────────────────────┐
│  Extract question features                     │
│  Score agents via FFN weight matrix             │
│  Three modes: full (≥0.6), compressed (0.3–0.6),│
│  skipped (<0.3).                                │
│  Mandatory (always full): SQL_DEVELOPER,        │
│  QUALITY_AUDITOR                                │
└──────────────────────────────────────────────┘
     │
     ▼
┌─ ADAPTIVE EVIDENCE FILTER ────────────────────┐
│  Retrieve top-k=5 chunks                       │
│  Deduplicate (Jaccard > 0.92)                  │
│  Relevance threshold (> 0.35)                  │
│  Self-assessed complexity: mean_sim < 0.57 →   │
│    "complex"                                    │
│  v3: Budget scales with retrieval similarity    │
│    sim≥0.61→500tok | ≥0.58→900 | ≥0.55→1200    │
│    <0.55→1500tok (full budget)                  │
│  └─ TIER 0: simple<0.40, complex<0.30 → ABSTAIN│
└──────────────────────────────────────────────┘
     │
     ▼
┌─ SELECTED AGENTS (pre-SQL) ───────────────────┐
│  DOMAIN_EXPERT, DATA_ENGINEER (if activated)    │
│  Communication pruning: each receives only      │
│  relevant prior outputs, not all                │
└──────────────────────────────────────────────┘
     │
     ▼
┌─ SQL_DEVELOPER ───────────────────────────────┐
│  Generate & execute SQL (retry up to 3x)        │
│  Attempt 3: ESCALATED (full agent outputs)      │
│  └─ TIER 1: FAILED → abstain                   │
└──────────────────────────────────────────────┘
     │
     ▼
┌─ TEMPORAL RE-SCORING ─────────────────────────┐
│  Adjust agent scores based on SQL outcome       │
│  └─ TIER 2: scalar result → skip QUANT_ANALYST │
│  Re-select post-SQL agents                      │
└──────────────────────────────────────────────┘
     │
     ▼
┌─ POST-SQL AGENTS ────────────────────────────┐
│  QUANTITATIVE_ANALYST (if not scalar-bypassed)  │
│  QUALITY_AUDITOR (always, validates answer)     │
└──────────────────────────────────────────────┘
     │
     ▼
  Final Answer
```

## Properties
- Same 5 expert agents and retrieval as MAS_RAG, plus adaptive mechanisms:
  1. **FFN gating (v2)**: 3-mode soft gating — full (≥0.6), compressed (0.3–0.6), skipped (<0.3)
  2. **Mandatory agents**: SQL_DEVELOPER + QUALITY_AUDITOR only (minimum viable pipeline)
  - 5-dim feature vector [1, d, m, r, c] — all keyword-based, no evaluation metadata
  3. **Self-assessed complexity**: retrieval similarity scores infer complexity (no human labels)
  4. **Adaptive evidence budget (v3)**: similarity-based budget (decoupled from agent count)
  5. **Communication pruning**: each agent receives only relevant prior outputs
  6. **Structured communication (v4)**: AgentMessage envelopes with parsed tables/joins/claims
  7. **Compressed mode (v2)**: borderline agents output structured JSON (~80% fewer tokens)
  8. **Escalated SQL retry**: attempt 3 bypasses pruning (full agent outputs)
  9. **Temporal re-scoring**: agent scores adjusted based on SQL outcome (modes updated too)
  10. **SQL_SAFETY_RULES** enforced in SQL_DEVELOPER prompt (same rules as all architectures)
  11. **SQL extraction guard**: extraction failure triggers retry (not raw response as SQL)
  12. **Early stopping** (3 tiers): evidence gate (Tier 0), SQL failed (Tier 1), scalar bypass (Tier 2)
  13. **Error handling**: technical errors → None answer; experiment loop try/except per question

In [0]:
import time
import json
import re
import uuid
import numpy as np
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Set
import pandas as pd

# ============================================================
# CONFIGURATION — Uses same AGENT_PROFILES from mt_config
# ============================================================
ARCHITECTURE = "DYNAMIC_FILTERED_MAS_RAG"

# LLM client — shared from mt_config (429-resilient, max_retries=5)
client = LLM_CLIENT

MODEL_ENDPOINT = MT_MODEL_ENDPOINT

# Build base context (technical schema only — semantic retrieved via RAG)
schema_context_dict = build_unified_context(ARCHITECTURE.replace("DYNAMIC_FILTERED_", ""))
schema_context_str = format_unified_context_for_prompt(schema_context_dict, include_semantic=False)

# ============================================================
# FILTERING & EFFICIENCY PARAMETERS
# ============================================================
# Base config — used as ceiling. Actual budget is computed adaptively
# based on question complexity (number of FFN-selected agents).
FILTER_CONFIG = {
    "relevance_threshold": 0.35,        # min similarity to retain a chunk
    "dedup_similarity_threshold": 0.92, # chunks above this are near-duplicates
    "max_context_tokens": 1500,         # max token budget (complex questions)
    "min_evidence_chunks": 1,           # minimum chunks to retain
    "max_evidence_chunks": 5,           # maximum chunks after filtering
}

# Similarity-based evidence budget — maps mean retrieval similarity → budget
# Decoupled from agent count entirely. Uses the retrieval signal directly.
# Empirical basis (from 25-question evaluation):
#   easy:      avg_sim = 0.619  →  high similarity  →  tight budget (well-covered)
#   medium:    avg_sim = 0.588  →  moderate          →  moderate budget
#   hard:      avg_sim = 0.592  →  moderate          →  moderate budget
#   very_hard: avg_sim = 0.564  →  low similarity    →  full budget (sparse coverage)
# Thresholds are set at the midpoints between empirical cluster means.
# Scanned top-to-bottom; first match wins.
SIMILARITY_BUDGET_MAP = [
    (0.61,  {"max_context_tokens": 500,  "max_evidence_chunks": 2}),  # trivial: strong coverage
    (0.58,  {"max_context_tokens": 900,  "max_evidence_chunks": 3}),  # moderate: decent coverage
    (0.55,  {"max_context_tokens": 1200, "max_evidence_chunks": 4}),  # hard: sparse coverage
    (0.0,   {"max_context_tokens": 1500, "max_evidence_chunks": 5}),  # very hard: minimal coverage
]


# ============================================================
# SELF-ASSESSED COMPLEXITY — Retrieval-Based Inference
# ============================================================
# Instead of relying on a human-labeled "difficulty" field, the agent
# infers question complexity from its own retrieval scores.
# Empirically validated: complex questions get lower avg similarity
# because they require reasoning across concepts that no single chunk covers.
#   easy:      avg_sim = 0.619
#   medium:    avg_sim = 0.588
#   hard:      avg_sim = 0.592
#   very_hard: avg_sim = 0.564
# Threshold 0.57 separates simple (well-covered) from complex (sparse coverage).
# ============================================================
COMPLEXITY_THRESHOLD = 0.57  # below this → "complex" (full budget)


def infer_complexity(raw_chunks: List[Dict]) -> str:
    """
    Self-assess question complexity from retrieval similarity scores.
    Zero cost — retrieval already happened.

    Returns:
        "complex" if mean similarity < COMPLEXITY_THRESHOLD (sparse coverage)
        "simple"  if mean similarity >= COMPLEXITY_THRESHOLD (well-covered)
    """
    if not raw_chunks:
        return "complex"  # no evidence → assume complex (need full budget)
    mean_sim = sum(c.get('similarity_score', 0) for c in raw_chunks) / len(raw_chunks)
    return "complex" if mean_sim < COMPLEXITY_THRESHOLD else "simple"


def get_adaptive_filter_config(mean_similarity: float, **_kwargs) -> Dict:
    """
    v3: Return evidence filter config based on mean retrieval similarity.

    Uses SIMILARITY_BUDGET_MAP for a continuous, difficulty-aware budget
    that is independent of agent count. Scans thresholds top-to-bottom;
    first match wins.

    Args:
        mean_similarity: mean cosine similarity of retrieved chunks (0–1).
                         Higher = better coverage = tighter budget allowed.
        **_kwargs:       ignored (backward compat with old callers).

    Returns:
        dict: merged FILTER_CONFIG with adaptive max_context_tokens and max_evidence_chunks.
    """
    # Scan SIMILARITY_BUDGET_MAP (descending thresholds); first match wins
    budget = SIMILARITY_BUDGET_MAP[-1][1]  # default: full budget
    for threshold, tier_budget in SIMILARITY_BUDGET_MAP:
        if mean_similarity >= threshold:
            budget = tier_budget
            break

    config = FILTER_CONFIG.copy()
    config.update(budget)
    return config

# ============================================================
# TEMPORAL FEED-FORWARD NETWORK (FFN) — Agent Gating
# ============================================================
# Heuristic scoring function that mimics a learned FFN.
# Input: question features  →  Output: per-agent relevance scores
# Agents below FFN_THRESHOLD are skipped.
# "Temporal" aspect: scores are re-evaluated after SQL execution.
# ============================================================

FFN_THRESHOLD = 0.6       # v2: agents scoring >= this run in FULL mode
FFN_COMPRESSED_THRESHOLD = 0.3  # v2: agents scoring >= this (but < 0.6) run in COMPRESSED mode
                                # agents scoring < 0.3 are SKIPPED entirely

# v2: Compressed mode — forces structured JSON output instead of free-text
# This preserves the agent's key insights at ~80% fewer tokens.
COMPRESSED_SYSTEM_PROMPT_TEMPLATE = """You are {role_name}, operating in compressed mode.
Provide ONLY a brief structured JSON analysis. Respond with valid JSON and nothing else.
Format:
{{
  "key_findings": ["finding1", "finding2"],
  "tables_identified": ["table1", "table2"],
  "join_keys": {{"table1.col": "table2.col"}},
  "domain_terms": {{"term": "meaning"}},
  "confidence": 0.8
}}
Keep each finding under 20 words. Maximum 3 findings. Be precise and factual."""

# Feature-to-agent weight matrix
# Each row: [base_score, domain_term, multi_table, rate_calc, comparison]
# 5-dimensional vector — no evaluation metadata (h removed to avoid data leakage).
# Design: base scores are low so that a SINGLE feature alone rarely activates an agent.
# Agents need 2+ relevant features to exceed threshold — this creates meaningful
# differentiation between simple (2 agents) and complex (5 agents) questions.
FFN_WEIGHTS = {
    "DOMAIN_EXPERT":         [0.15, 0.35, 0.10, 0.05, 0.05],
    "DATA_ENGINEER":         [0.15, 0.00, 0.40, 0.00, 0.05],
    "SQL_DEVELOPER":         [0.85, 0.00, 0.05, 0.00, 0.00],   # mandatory
    "QUANTITATIVE_ANALYST":  [0.10, 0.00, 0.00, 0.40, 0.25],
    "QUALITY_AUDITOR":       [0.80, 0.00, 0.00, 0.00, 0.00],   # mandatory
}

# Temporal re-scoring: after SQL execution, adjust scores
TEMPORAL_ADJUSTMENTS = {
    # If SQL succeeded with simple result (few rows), reduce QUANTITATIVE need
    "sql_success_simple": {"QUANTITATIVE_ANALYST": -0.2},
    # If SQL failed, boost QUALITY_AUDITOR (to properly abstain)
    "sql_failed": {"QUALITY_AUDITOR": +0.2},
    # If SQL returned complex multi-row result, boost QUANTITATIVE
    "sql_success_complex": {"QUANTITATIVE_ANALYST": +0.3},
}

# Max SQL retries — attempt 3 uses escalated context (full agent outputs)
# to compensate for pruned context in attempts 1-2.
MAX_SQL_RETRIES = 3

# ============================================================
# AGENT POOL — Uses AGENT_PROFILES from mt_config (5 experts)
# All candidates; FFN decides which to activate per question.
# ============================================================
# ============================================================
# FFN SCORING FUNCTION
# ============================================================

def extract_question_features(question_dict: Dict) -> List[float]:
    """
    Extract 5-dimensional feature vector from question text only.
    All features are detected via deterministic keyword matching — no
    evaluation metadata is used, so the vector is inference-safe.
    Returns: [base=1.0, domain_terms, multi_table, rate_calc, comparison]
    """
    question = question_dict.get("question", "")
    q_lower = question.lower()

    # Feature 1: Domain-specific terms (German business terminology)
    domain_terms = any(term in q_lower for term in [
        "pflicht", "wahl", "mandatory", "elective", "compliance",
        "on-time", "deadline", "certificate", "zertifikat",
        "company type", "baercare", "care", "mvz"
    ])

    # Feature 2: Multi-table join likely needed (keyword-only, no metadata)
    multi_table = any(term in q_lower for term in [
        "company", "assignment", "by company", "per company", "company type"
    ])

    # Feature 3: Rate/percentage calculation
    rate_calc = any(term in q_lower for term in [
        "rate", "percentage", "percent", "%", "ratio", "proportion",
        "how many", "share", "fraction"
    ])

    # Feature 4: Comparison/trend analysis
    comparison = any(term in q_lower for term in [
        "compare", "versus", "vs", "trend", "growth", "decline",
        "month-over-month", "drop", "increase", "decrease", "change",
        "highest", "lowest", "top", "ranking", "best", "worst"
    ])

    return [1.0, float(domain_terms), float(multi_table), float(rate_calc),
            float(comparison)]


def ffn_score_agents(features: List[float]) -> Dict[str, float]:
    """
    Temporal FFN forward pass: compute relevance score for each agent.
    score(agent) = dot_product(features, weights[agent])
    Clipped to [0, 1].
    """
    scores = {}
    for agent_name, weights in FFN_WEIGHTS.items():
        score = sum(f * w for f, w in zip(features, weights))
        scores[agent_name] = max(0.0, min(1.0, score))
    return scores


def select_agents(scores: Dict[str, float], threshold: float = None) -> List[str]:
    """
    v2: Soft gating — select all agents that are not SKIPPED.
    Agents with score >= FFN_COMPRESSED_THRESHOLD (0.3) are included.
    Their operating mode (full vs compressed) is determined by get_agent_modes().
    Mandatory agents are always included regardless of score.
    """
    mandatory = {"SQL_DEVELOPER", "QUALITY_AUDITOR"}
    selected = set()

    for agent_name, score in scores.items():
        # Include if mandatory OR if score is above the skip threshold
        if agent_name in mandatory or score >= FFN_COMPRESSED_THRESHOLD:
            selected.add(agent_name)

    # Return in canonical order
    return [a for a in MAS_AGENT_ORDER if a in selected]


def get_agent_modes(scores: Dict[str, float]) -> Dict[str, str]:
    """
    v2: Determine operating mode for each agent based on FFN score.

    Three modes:
      - "full":       score >= 0.6  OR mandatory agent → normal free-text analysis
      - "compressed": 0.3 <= score < 0.6 → structured JSON output only (~80% fewer tokens)
      - "skipped":    score < 0.3 → agent does not execute

    Returns: dict mapping agent_name → mode string
    """
    mandatory = {"SQL_DEVELOPER", "QUALITY_AUDITOR"}
    modes = {}
    for agent_name, score in scores.items():
        if agent_name in mandatory:
            modes[agent_name] = "full"
        elif score >= FFN_THRESHOLD:
            modes[agent_name] = "full"
        elif score >= FFN_COMPRESSED_THRESHOLD:
            modes[agent_name] = "compressed"
        else:
            modes[agent_name] = "skipped"
    return modes


def temporal_rescore(scores: Dict[str, float], sql_status: str, n_result_rows: int) -> Dict[str, float]:
    """
    Temporal re-scoring after SQL execution.
    Adjusts agent scores based on SQL outcome for downstream agents.
    """
    adjusted = scores.copy()

    if sql_status == "failed":
        adjustments = TEMPORAL_ADJUSTMENTS.get("sql_failed", {})
    elif sql_status == "success" and n_result_rows <= 2:
        adjustments = TEMPORAL_ADJUSTMENTS.get("sql_success_simple", {})
    elif sql_status == "success" and n_result_rows > 2:
        adjustments = TEMPORAL_ADJUSTMENTS.get("sql_success_complex", {})
    else:
        adjustments = {}

    for agent, delta in adjustments.items():
        if agent in adjusted:
            adjusted[agent] = max(0.0, min(1.0, adjusted[agent] + delta))

    return adjusted


print(f"✓ {ARCHITECTURE} configured")
print(f"  Expert pool: {list(AGENT_PROFILES.keys())}")
print(f"  FFN gating: full>={FFN_THRESHOLD}, compressed>={FFN_COMPRESSED_THRESHOLD}, skipped<{FFN_COMPRESSED_THRESHOLD}")
print(f"  Mandatory agents (always full): SQL_DEVELOPER, QUALITY_AUDITOR")
print(f"  Model: {MODEL_ENDPOINT}")
print(f"  Evidence filter: relevance>{FILTER_CONFIG['relevance_threshold']}, dedup>{FILTER_CONFIG['dedup_similarity_threshold']}")
print(f"  Adaptive budget (sim-based): sim>=0.61→500tok/2ch | >=0.58→900/3 | >=0.55→1200/4 | <0.55→1500/5")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7863973565040096>, line 22
     19 _token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
     20 client = OpenAI(api_key=_token, base_url=f"https://{workspace_url}/serving-endpoints")
---> 22 MODEL_ENDPOINT = MT_MODEL_ENDPOINT
     24 # Build base context (technical schema only — semantic retrieved via RAG)
     25 schema_context_dict = build_unified_context(ARCHITECTURE.replace("DYNAMIC_FILTERED_", ""))

NameError: name 'MT_MODEL_ENDPOINT' is not defined

In [0]:
# ============================================================
# EVIDENCE FILTERING MODULE
# ============================================================
# Remove redundant information to reduce token waste and
# prevent noise-induced hallucinations.
# ============================================================

@dataclass
class EvidenceItem:
    """Single piece of authoritative evidence with provenance."""
    evidence_id: str
    source_type: str          # 'semantic_chunk' | 'sql_result' | 'schema_info'
    source_reference: str     # table name or chunk source
    raw_content: str
    relevance_score: float    # similarity score from retrieval
    retained: bool = True
    token_count: int = 0
    filter_reason: Optional[str] = None


def estimate_tokens(text: str) -> int:
    """Rough token estimate (4 chars per token for English/SQL mix)."""
    return max(1, len(text) // 4)


def filter_evidence(
    chunks: List[Dict],
    config: Dict = None
) -> Tuple[List[EvidenceItem], List[EvidenceItem]]:
    """
    Filter retrieved evidence: deduplicate, relevance-threshold, token budget.

    Returns:
        (retained_evidence, dropped_evidence)
    """
    cfg = config or FILTER_CONFIG

    # Convert to EvidenceItems
    all_evidence = []
    for i, chunk in enumerate(chunks):
        content = chunk.get('content', chunk.get('text', str(chunk)))
        score = chunk.get('score', chunk.get('relevance_score', 0.5))
        all_evidence.append(EvidenceItem(
            evidence_id=f"ev_{i:03d}",
            source_type="semantic_chunk",
            source_reference=chunk.get('source', chunk.get('metadata', {}).get('source', 'unknown')),
            raw_content=content,
            relevance_score=score,
            token_count=estimate_tokens(content),
        ))

    # Sort by relevance (highest first)
    all_evidence.sort(key=lambda e: e.relevance_score, reverse=True)

    retained = []
    dropped = []
    total_tokens = 0

    for ev in all_evidence:
        # Step 1: Relevance threshold
        if ev.relevance_score < cfg["relevance_threshold"]:
            ev.retained = False
            ev.filter_reason = f"below_relevance_threshold ({ev.relevance_score:.3f} < {cfg['relevance_threshold']})"
            dropped.append(ev)
            continue

        # Step 2: Near-duplicate detection (compare with retained)
        is_duplicate = False
        for retained_ev in retained:
            # Simple content overlap check (jaccard on words)
            words_a = set(ev.raw_content.lower().split())
            words_b = set(retained_ev.raw_content.lower().split())
            if len(words_a) > 0 and len(words_b) > 0:
                jaccard = len(words_a & words_b) / len(words_a | words_b)
                if jaccard > cfg["dedup_similarity_threshold"]:
                    is_duplicate = True
                    break
        if is_duplicate:
            ev.retained = False
            ev.filter_reason = "near_duplicate"
            dropped.append(ev)
            continue

        # Step 3: Token budget
        if total_tokens + ev.token_count > cfg["max_context_tokens"] and len(retained) >= cfg["min_evidence_chunks"]:
            ev.retained = False
            ev.filter_reason = f"token_budget_exceeded ({total_tokens + ev.token_count} > {cfg['max_context_tokens']})"
            dropped.append(ev)
            continue

        # Step 4: Max chunks
        if len(retained) >= cfg["max_evidence_chunks"]:
            ev.retained = False
            ev.filter_reason = f"max_chunks_reached ({cfg['max_evidence_chunks']})"
            dropped.append(ev)
            continue

        # Retain
        retained.append(ev)
        total_tokens += ev.token_count

    return retained, dropped


def format_filtered_evidence(evidence: List[EvidenceItem]) -> str:
    """Format retained evidence for LLM prompts with evidence IDs."""
    if not evidence:
        return "(no relevant evidence found)"
    lines = ["RETRIEVED EVIDENCE (filtered):"]
    for ev in evidence:
        lines.append(f"  [{ev.evidence_id}] (score={ev.relevance_score:.2f}) {ev.raw_content[:300]}")
    return "\n".join(lines)


# ============================================================
# STRUCTURED COMMUNICATION — Message envelopes between agents
# ============================================================

@dataclass
class AgentMessage:
    """Structured message passed between agents (memory-size-one: only latest kept)."""
    sender: str
    receiver: str
    status: str               # SUCCESS | FAILED | ABSTAINED | NEEDS_EXPANSION
    claims: List[Dict] = field(default_factory=list)  # [{"claim": str, "evidence_ids": [str]}]
    tables_identified: List[str] = field(default_factory=list)  # v4: tables mentioned
    join_keys: Dict[str, str] = field(default_factory=dict)     # v4: join relationships
    sql_query: Optional[str] = None
    sql_results: Optional[str] = None
    plan: Optional[str] = None
    missing_information: List[str] = field(default_factory=list)
    uncertainty: List[str] = field(default_factory=list)
    needs_expansion: bool = False
    token_count: int = 0

    def to_prompt_str(self) -> str:
        """Serialize for LLM prompt (compact, no redundancy)."""
        parts = [f"FROM: {self.sender} | STATUS: {self.status}"]
        if self.plan:
            parts.append(f"PLAN: {self.plan[:300]}")
        if self.tables_identified:
            parts.append(f"TABLES: {', '.join(self.tables_identified)}")
        if self.join_keys:
            jk = '; '.join(f'{k}={v}' for k, v in self.join_keys.items())
            parts.append(f"JOINS: {jk}")
        if self.sql_query:
            parts.append(f"SQL: {self.sql_query}")
        if self.sql_results:
            parts.append(f"RESULTS:\n{self.sql_results[:400]}")
        if self.claims:
            claims_str = "; ".join(c.get('claim', '') for c in self.claims[:5])
            parts.append(f"CLAIMS: {claims_str}")
        if self.missing_information:
            parts.append(f"MISSING: {', '.join(self.missing_information)}")
        result = "\n".join(parts)
        self.token_count = estimate_tokens(result)
        return result


# ============================================================
# v4: RESPONSE PARSER — Extract structure from free-text output
# ============================================================
# Known table names from the LMS dataset (used to detect table references)
_KNOWN_TABLES = {
    "mt_safe_user", "mt_safe_company", "mt_safe_assignment",
    "mt_safe_course", "mt_safe_assignment_course_progress", "mt_safe_company_type",
}
# Common join key patterns
_JOIN_KEY_PATTERN = re.compile(
    r'(\w+)\s*(?:=|JOIN\s+ON|joined?\s+(?:on|via|using|by))\s*(\w+)',
    re.IGNORECASE
)


def parse_agent_response(response: str, agent_name: str) -> AgentMessage:
    """
    v4: Parse an agent's free-text response into a structured AgentMessage.

    Strategy: extract what we can (tables, join keys, claims/findings,
    missing info) via pattern matching. Falls back gracefully — if nothing
    is extractable, the plan field carries a truncated summary.

    Works for both full-mode free-text AND compressed-mode JSON.
    """
    msg = AgentMessage(sender=agent_name, receiver="NEXT", status="SUCCESS")

    # ---- Try JSON parse first (compressed mode returns JSON) ----
    try:
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            msg.claims = [{"claim": f} for f in parsed.get("key_findings", [])]
            msg.tables_identified = parsed.get("tables_identified", [])
            raw_jk = parsed.get("join_keys", {})
            msg.join_keys = {str(k): str(v) for k, v in raw_jk.items()} if isinstance(raw_jk, dict) else {}
            msg.plan = "; ".join(parsed.get("key_findings", []))[:300]
            if parsed.get("domain_terms"):
                terms = parsed["domain_terms"]
                if isinstance(terms, dict):
                    msg.plan = (msg.plan or "") + " | Terms: " + "; ".join(f"{k}={v}" for k, v in terms.items())
            return msg
    except (json.JSONDecodeError, TypeError):
        pass  # Not JSON — parse as free text

    # ---- Free-text parsing ----
    resp_lower = response.lower()

    # Extract tables: match known table names in the response
    for tbl in _KNOWN_TABLES:
        if tbl in resp_lower:
            msg.tables_identified.append(tbl)

    # Extract join keys: look for "table.column = table.column" patterns
    join_matches = re.findall(r'(\w+\.\w+)\s*=\s*(\w+\.\w+)', response)
    for left, right in join_matches[:5]:
        msg.join_keys[left] = right

    # Extract claims: sentences containing data assertions
    _claim_signals = ['%', 'total', 'count', 'average', 'rate', 'number of',
                      'most', 'highest', 'lowest', 'ratio', 'proportion']
    sentences = re.split(r'[.\n]', response)
    for sent in sentences:
        sent = sent.strip()
        if len(sent) > 15 and any(sig in sent.lower() for sig in _claim_signals):
            msg.claims.append({"claim": sent[:150]})
            if len(msg.claims) >= 5:
                break

    # Extract missing information
    _missing_signals = ['not available', 'no data', 'cannot determine',
                        'missing', 'not found', 'insufficient', 'no table']
    for sent in sentences:
        if any(sig in sent.lower() for sig in _missing_signals):
            msg.missing_information.append(sent.strip()[:100])
            if len(msg.missing_information) >= 3:
                break

    # Plan: first 300 chars of the response as a summary
    msg.plan = response[:300].replace('\n', ' ').strip()

    # Check for abstention signals
    if 'CANNOT_ANSWER' in response.upper() or 'INSUFFICIENT_EVIDENCE' in response.upper():
        msg.status = "ABSTAINED"

    return msg


def parse_sql_agent_response(state) -> AgentMessage:
    """
    v4: Build AgentMessage for SQL_DEVELOPER from workflow state.
    SQL agent has special handling: query and results are tracked in state.
    """
    msg = AgentMessage(
        sender="SQL_DEVELOPER",
        receiver="NEXT",
        status=state.sql_execution_status.upper() if state.sql_execution_status else "UNKNOWN",
        sql_query=state.sql_generated,
        sql_results=state.sql_results,
    )
    # Extract tables from SQL
    if state.sql_generated:
        sql_lower = state.sql_generated.lower()
        for tbl in _KNOWN_TABLES:
            if tbl in sql_lower:
                msg.tables_identified.append(tbl)
    return msg


print("✓ Evidence filtering & structured communication defined")
print(f"  Relevance threshold: {FILTER_CONFIG['relevance_threshold']}")
print(f"  Dedup threshold: {FILTER_CONFIG['dedup_similarity_threshold']}")
print(f"  Token budget: {FILTER_CONFIG['max_context_tokens']}")

✓ Evidence filtering & structured communication defined
  Relevance threshold: 0.35
  Dedup threshold: 0.92
  Token budget: 1500


In [0]:
# ============================================================
# DYNAMIC WORKFLOW STATE & AGENT RUNNER
# ============================================================
# Uses same run_agent() pattern as MAS_RAG but with:
# 1. FFN-gated agent selection (not all 5 always run)
# 2. Temporal re-scoring after SQL execution
# 3. Evidence filtering before agent pipeline
# ============================================================

from dataclasses import dataclass, field

@dataclass
class DynamicWorkflowState:
    """Extended workflow state for FFN-gated dynamic pipeline."""
    question: str = ""
    question_id: str = ""
    # FFN gating
    features: List[float] = field(default_factory=list)
    initial_scores: Dict[str, float] = field(default_factory=dict)
    temporal_scores: Dict[str, float] = field(default_factory=dict)
    selected_agents: List[str] = field(default_factory=list)
    skipped_agents: List[str] = field(default_factory=list)
    agent_modes: Dict[str, str] = field(default_factory=dict)  # v2: agent_name → "full"|"compressed"|"skipped"
    # Evidence filtering
    raw_chunks: List[Dict] = field(default_factory=list)
    retained_evidence: List = field(default_factory=list)
    dropped_evidence: List = field(default_factory=list)
    filtered_context_str: str = ""
    # Agent contributions
    agent_outputs: Dict[str, str] = field(default_factory=dict)
    # SQL execution
    sql_generated: Optional[str] = None
    sql_results: Optional[str] = None
    sql_execution_status: str = "not_attempted"
    sql_attempts: int = 0
    sql_result_rows: int = 0
    # Final output
    final_answer: Optional[str] = None
    can_answer: bool = True
    # Orchestration metadata
    steps_log: List[Dict] = field(default_factory=list)
    total_prompt_tokens: int = 0
    total_completion_tokens: int = 0
    llm_calls: int = 0
    status: str = "in_progress"
    error: Optional[str] = None


# ============================================================
# COMMUNICATION ROUTING — Selective message passing
# ============================================================
# Each agent receives ONLY the outputs of agents relevant to its task.
# This reduces prompt size for later agents and prevents noise.
# SQL results are passed via a dedicated block (not affected by this routing).
# ============================================================
COMMUNICATION_ROUTING = {
    "DOMAIN_EXPERT":        [],                                      # first agent — no prior context
    "DATA_ENGINEER":        ["DOMAIN_EXPERT"],                       # needs business interpretation to find tables
    "SQL_DEVELOPER":        ["DOMAIN_EXPERT", "DATA_ENGINEER"],      # needs what to compute + how to join
    "QUANTITATIVE_ANALYST": ["DOMAIN_EXPERT"],                       # needs domain context for interpretation
    "QUALITY_AUDITOR":      ["QUANTITATIVE_ANALYST"],                # needs claims to validate against SQL results
}


# v4: Structured output suffix — appended to full-mode agent prompts.
# Asks the agent to include structured markers without constraining its analysis.
# This makes parse_agent_response() more reliable without forcing pure JSON.
STRUCTURED_OUTPUT_SUFFIX = """

At the END of your analysis, include these structured markers:
TABLES_USED: [comma-separated table names]
JOIN_KEYS: [table1.col = table2.col, ...]
KEY_CLAIMS: [numbered list of 3-5 factual claims, one per line]
MISSING_INFO: [what data is unavailable, if any]
"""


def _build_dynamic_context(state: DynamicWorkflowState, current_agent: str) -> str:
    """
    v4: Build selective context from structured AgentMessage outputs.
    Only includes outputs from agents listed in COMMUNICATION_ROUTING[current_agent].
    Agents not in the routing map (or skipped by FFN) are excluded.
    Now passes compact to_prompt_str() representations (~100-300 tokens each)
    instead of raw free-text (~500-1500 tokens each).
    """
    if not state.agent_outputs:
        return ""

    # Get the list of agents whose output this agent should see
    allowed_sources = COMMUNICATION_ROUTING.get(current_agent, [])

    parts = []
    for agent_name in allowed_sources:
        if agent_name in state.agent_outputs:
            parts.append(f"--- {agent_name} ---\n{state.agent_outputs[agent_name]}")

    return "\n\n".join(parts) if parts else ""


def run_dynamic_agent(state: DynamicWorkflowState, agent_name: str) -> DynamicWorkflowState:
    """
    Run a single expert agent in the dynamic pipeline.
    Same logic as MAS_RAG's run_agent() but uses filtered evidence context.
    """
    if not state.can_answer:
        return state

    profile = AGENT_PROFILES[agent_name]
    step_start = time.time()
    step_idx = len(state.steps_log) + 1

    collab_context = _build_dynamic_context(state, agent_name)

    base_context = (
        f"QUESTION: {state.question}\n\n"
        f"SCHEMA CONTEXT:\n{schema_context_str}\n\n"
        f"{state.filtered_context_str}\n"
    )

    if collab_context:
        base_context += f"\nPREVIOUS AGENTS' ANALYSIS:\n{collab_context}\n"

    # Add SQL results for post-SQL agents
    if state.sql_generated and agent_name in ["QUANTITATIVE_ANALYST", "QUALITY_AUDITOR"]:
        base_context += (
            f"\nSQL QUERY EXECUTED:\n{state.sql_generated}\n"
            f"SQL STATUS: {state.sql_execution_status}\n"
            f"SQL RESULTS:\n{state.sql_results or '(empty)'}\n"
        )

    # --- SQL_DEVELOPER: execute + retry ---
    # Attempt 1: normal pruned context (fast path)
    # Attempt 2: error feedback + same context (simple fix)
    # Attempt 3: ESCALATED — full agent outputs bypass pruning (complex fix)
    if agent_name == "SQL_DEVELOPER":
        for attempt in range(1, MAX_SQL_RETRIES + 1):
            state.sql_attempts = attempt
            retry_context = ""
            if attempt > 1 and state.sql_execution_status == "failed":
                retry_context = (
                    f"\n\nPREVIOUS SQL ATTEMPT FAILED (attempt {attempt-1}):\n"
                    f"SQL: {state.sql_generated}\n"
                    f"Error: {state.sql_results}\n"
                    f"Fix the query and try again."
                )
                # ESCALATED RETRY: on attempt 3+, bypass communication pruning
                # Give SQL_DEVELOPER ALL available agent outputs (mimics MAS_RAG broadcast)
                if attempt >= 3 and state.agent_outputs:
                    escalated_parts = []
                    for src_agent, src_output in state.agent_outputs.items():
                        if src_agent != "SQL_DEVELOPER":
                            escalated_parts.append(f"--- {src_agent} ---\n{src_output}")
                    if escalated_parts:
                        retry_context += (
                            f"\n\nESCALATED CONTEXT (all agent outputs for complex fix):\n"
                            + "\n\n".join(escalated_parts)
                        )

            _sql_rules = "\n".join(f"- {r}" for r in SQL_SAFETY_RULES)
            user_prompt = base_context + retry_context + f"\n\nSQL RULES:\n{_sql_rules}\n\nWrite the SQL query now:"

            response, pt, ct = _call_llm(profile["system_prompt"], user_prompt, profile["llm_params"])
            state.total_prompt_tokens += pt
            state.total_completion_tokens += ct
            state.llm_calls += 1

            if "NO_SQL_NEEDED" in response.upper() or "CANNOT_ANSWER" in response.upper():
                state.sql_execution_status = "not_applicable"
                _msg = AgentMessage(sender="SQL_DEVELOPER", receiver="NEXT", status="NOT_APPLICABLE")
                _msg.plan = response[:200]
                state.agent_outputs[agent_name] = _msg.to_prompt_str()
                break

            state.sql_generated = extract_sql_from_response(response)
            if not state.sql_generated:
                # Extraction failed — treat as SQL failure for retry
                state.sql_execution_status = "failed"
                state.sql_results = "SQL ERROR: Could not extract valid SQL from LLM response"
                continue  # retry with error feedback
            
            state.sql_results = execute_sql_query(state.sql_generated)

            if state.sql_results.startswith("SQL ERROR"):
                state.sql_execution_status = "failed"
            elif state.sql_results.strip() == '' or 'Empty DataFrame' in state.sql_results:
                state.sql_execution_status = "success"
                state.sql_result_rows = 0
                _msg = parse_sql_agent_response(state)
                state.agent_outputs[agent_name] = _msg.to_prompt_str()
                break
            else:
                state.sql_execution_status = "success"
                state.sql_result_rows = state.sql_results.count("\n")
                # v4: structured SQL output for downstream agents
                _msg = parse_sql_agent_response(state)
                state.agent_outputs[agent_name] = _msg.to_prompt_str()
                break
        else:
            _msg = AgentMessage(sender="SQL_DEVELOPER", receiver="NEXT", status="FAILED",
                                sql_query=state.sql_generated, sql_results=state.sql_results)
            state.agent_outputs[agent_name] = _msg.to_prompt_str()

    # --- All other agents ---
    else:
        # v2: Soft gating — check agent mode (full vs compressed)
        _mode = state.agent_modes.get(agent_name, "full")

        if _mode == "compressed":
            # COMPRESSED MODE: structured JSON output, reduced token budget
            _compressed_sys = COMPRESSED_SYSTEM_PROMPT_TEMPLATE.format(
                role_name=profile['description']
            )
            user_prompt = (
                f"QUESTION: {state.question}\n\n"
                f"SCHEMA CONTEXT (abbreviated):\n{schema_context_str[:2000]}\n\n"
                f"{state.filtered_context_str}\n"
                f"\nRespond with JSON only."
            )
            _compressed_params = {"temperature": 0.1, "max_tokens": 250}
            response, pt, ct = _call_llm(_compressed_sys, user_prompt, _compressed_params)
        else:
            # FULL MODE: free-text analysis + structured output suffix (v4)
            _sys_prompt = profile["system_prompt"]
            if agent_name != "QUALITY_AUDITOR":
                # Add structured markers suffix for non-final agents
                # QUALITY_AUDITOR is the final output agent — its response
                # is user-facing and should not be constrained.
                _sys_prompt = _sys_prompt + STRUCTURED_OUTPUT_SUFFIX
            user_prompt = base_context + f"\nProvide your expert analysis as {profile['description']}:"
            response, pt, ct = _call_llm(_sys_prompt, user_prompt, profile["llm_params"])

        state.total_prompt_tokens += pt
        state.total_completion_tokens += ct
        state.llm_calls += 1

        # v4: Parse response into structured AgentMessage for inter-agent communication
        _agent_msg = parse_agent_response(response, agent_name)
        # Store the compact structured version for downstream agents
        state.agent_outputs[agent_name] = _agent_msg.to_prompt_str()

        # Early termination for non-QA agents only.
        # QUALITY_AUDITOR has its own startswith logic below — the generic
        # check must NOT fire for QA, or it corrupts sql_execution_status.
        if "CANNOT_ANSWER" in response.upper() and agent_name != "QUALITY_AUDITOR":
            state.can_answer = False
            state.status = "abstained"
            state.final_answer = response  # keep raw response for user output
            state.sql_execution_status = "not_applicable"

        if agent_name == "QUALITY_AUDITOR":
            # QUALITY_AUDITOR is the final output — preserve raw response
            state.final_answer = response
            # Only abstain when the response STARTS with CANNOT_ANSWER
            # (meaning the auditor determined the schema cannot support the question).
            # INSUFFICIENT_EVIDENCE and HALLUCINATION_DETECTED should NOT trigger
            # abstention — the auditor should still produce a best-effort answer.
            response_trimmed = response.strip().upper()
            if response_trimmed.startswith("CANNOT_ANSWER"):
                state.status = "abstained"
            else:
                state.status = "success"

    step_latency = time.time() - step_start
    state.steps_log.append({
        "step": step_idx,
        "agent": agent_name,
        "role": profile["role"],
        "mode": state.agent_modes.get(agent_name, "full"),  # v2/v4: full|compressed
        "status": state.sql_execution_status.upper() if agent_name == "SQL_DEVELOPER" else "SUCCESS",
        "latency_ms": round(step_latency * 1000, 1),
        "ffn_score": state.initial_scores.get(agent_name, 0),
    })
    return state


print("✓ Dynamic agent runner defined (v4: structured AgentMessage communication)")
print(f"  run_dynamic_agent(state, agent_name)")
print(f"  Temporal re-scoring after SQL execution")
print(f"  Inter-agent messages: parse_agent_response() → AgentMessage.to_prompt_str()")
print(f"  Full-mode agents get STRUCTURED_OUTPUT_SUFFIX for reliable parsing")
print(f"  QUALITY_AUDITOR final_answer preserved as raw free-text")  


✓ Dynamic agent runner defined
  run_dynamic_agent(state, agent_name)
  Temporal re-scoring after SQL execution


In [0]:
# ============================================================
# CORE UTILITIES
# ============================================================

def execute_sql_query(sql: str) -> str:
    """Execute SQL and return results. Reuse existing pattern."""
    try:
        result_df = spark.sql(sql).toPandas()
        n_rows = len(result_df)
        if n_rows > 30:
            return f"({n_rows} rows total, showing first 30)\n" + result_df.head(30).to_string(index=False)
        return result_df.to_string(index=False)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"


def extract_sql_from_response(text: str) -> Optional[str]:
    """Extract SQL query from LLM response."""
    match = re.search(r"```(?:sql)?\s*(.+?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    lines = text.strip().split("\n")
    sql_lines = []
    capture = False
    for line in lines:
        stripped = line.strip().upper()
        if stripped.startswith(("SELECT", "WITH")):
            capture = True
        if capture:
            sql_lines.append(line)
    if sql_lines:
        return "\n".join(sql_lines).strip()
    if any(kw in text.upper() for kw in ["SELECT", "FROM"]):
        return text.strip()
    return None


def _call_llm(system_prompt: str, user_prompt: str, params: dict) -> Tuple[str, int, int]:
    """Single LLM call. Returns (text, prompt_tokens, completion_tokens)."""
    resp = client.chat.completions.create(
        model=MODEL_ENDPOINT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=params["temperature"],
        max_tokens=params["max_tokens"],
        **_seed_kwargs()
    )
    return (
        resp.choices[0].message.content,
        resp.usage.prompt_tokens,
        resp.usage.completion_tokens
    )


print("✓ Core utilities defined: execute_sql_query, extract_sql_from_response, _call_llm")

✓ Core utilities defined: execute_sql_query, extract_sql_from_response, _call_llm


In [0]:
# Internal signal for clean early-stop jumps (avoids deeply nested if/else)
class _EarlyStopSignal(Exception):
    pass


# ============================================================
# FFN-GATED ORCHESTRATOR WITH EARLY STOPPING
# ============================================================
# Core innovations:
# 1. FFN gating: selects which agents participate per question
# 2. Adaptive evidence budget: scales with complexity
# 3. Communication pruning: selective message routing
# 4. Early stopping (3 tiers):
#    - Tier 0: Evidence insufficiency → abstain (0 LLM calls)
#    - Tier 1: SQL EMPTY → deterministic "no data" (skip interpretation)
#    - Tier 2: Simple scalar → skip QUANTITATIVE_ANALYST
# ============================================================

# Early stopping configuration (difficulty-aware Tier 0)
EARLY_STOPPING_CONFIG = {
    "evidence_min_mean_score": 0.40,        # Tier 0: default threshold (easy/medium)
    "evidence_min_mean_score_hard": 0.30,   # Tier 0: relaxed threshold (hard/very_hard)
    "scalar_max_rows": 1,                   # Tier 2: max rows to classify as scalar
    "scalar_max_cols": 2,                   # Tier 2: max columns to classify as scalar
}


def _is_scalar_result(sql_results: str) -> bool:
    """
    Determine if SQL result is a simple scalar (≤1 data row, ≤2 columns).
    Parses the text representation from DataFrame.to_string(index=False).
    """
    if not sql_results or "SQL ERROR" in sql_results:
        return False
    lines = [l for l in sql_results.strip().split("\n") if l.strip()]
    if len(lines) < 2:
        return False  # need at least header + 1 data row
    # First line = header, remaining = data rows
    header = lines[0]
    data_rows = lines[1:]  # skip header
    n_cols = len(header.split())
    n_rows = len(data_rows)
    return (n_rows <= EARLY_STOPPING_CONFIG["scalar_max_rows"] and
            n_cols <= EARLY_STOPPING_CONFIG["scalar_max_cols"])


def run_dynamic_filtered_mas_rag(question_dict: Dict) -> Dict:
    """
    Execute DYNAMIC_FILTERED_MAS_RAG for one question.

    Flow:
    1. FFN forward pass → agent selection
    2. Retrieve + adaptive filter evidence
       └── TIER 0: evidence insufficient → abstain (0 LLM calls)
    3. Run pre-SQL agents (if selected)
    4. Run SQL_DEVELOPER
       └── TIER 1: SQL EMPTY → deterministic answer (skip post-SQL agents)
    5. Temporal re-scoring
       └── TIER 2: scalar result → skip QUANTITATIVE_ANALYST
    6. Run post-SQL agents
    7. Fallback
    """
    start_time = time.time()

    # Initialize state
    state = DynamicWorkflowState(
        question=question_dict["question"],
        question_id=question_dict["id"],
    )
    adaptive_config = FILTER_CONFIG.copy()  # default; overwritten after agent selection
    early_stop_tier = None  # tracks which tier triggered (for metadata)

    try:
        # ---- STEP 1: FFN FORWARD PASS → Agent Selection ----
        state.features = extract_question_features(question_dict)
        state.initial_scores = ffn_score_agents(state.features)
        state.selected_agents = select_agents(state.initial_scores)
        state.agent_modes = get_agent_modes(state.initial_scores)  # v2: soft gating modes
        state.skipped_agents = [a for a in MAS_AGENT_ORDER if state.agent_modes.get(a) == "skipped"]

        # v2: build mode summary for logging
        _mode_summary = {k: v for k, v in state.agent_modes.items() if v != "skipped"}

        state.steps_log.append({
            "step": 1, "agent": "FFN_ROUTER",
            "action": f"select_agents={state.selected_agents}",
            "status": "SUCCESS",
            "latency_ms": 0,
            "ffn_score": 0,
            "details": f"scores={{{', '.join(f'{k}:{v:.2f}' for k, v in state.initial_scores.items())}}}, modes={_mode_summary}, skipped={state.skipped_agents}",
        })

        # ---- STEP 2: RETRIEVE & SELF-ASSESS COMPLEXITY ----
        step_start = time.time()
        state.raw_chunks = retrieve_semantic_context(state.question)

        # SELF-ASSESSED COMPLEXITY: infer from retrieval scores (zero extra cost)
        # The agent determines its own evidence budget based on how well the
        # knowledge base covers the question — no human difficulty label needed.
        _inferred_complexity = infer_complexity(state.raw_chunks)
        _mean_retrieval_sim = sum(c.get('similarity_score', 0) for c in state.raw_chunks) / max(1, len(state.raw_chunks))

        # v3: Adaptive filtering — budget derived from retrieval similarity (not agent count)
        adaptive_config = get_adaptive_filter_config(_mean_retrieval_sim)
        state.retained_evidence, state.dropped_evidence = filter_evidence(state.raw_chunks, config=adaptive_config)
        state.filtered_context_str = format_filtered_evidence(state.retained_evidence)

        state.steps_log.append({
            "step": 2, "agent": "EVIDENCE_FILTER",
            "action": f"filter: {len(state.raw_chunks)}→{len(state.retained_evidence)} chunks (budget={adaptive_config['max_context_tokens']}tok, max={adaptive_config['max_evidence_chunks']})",
            "status": "SUCCESS",
            "latency_ms": round((time.time() - step_start) * 1000, 1),
            "ffn_score": 0,
            "details": f"self-assessed: mean_sim={_mean_retrieval_sim:.3f} → {_inferred_complexity} | sim-based budget → {adaptive_config['max_context_tokens']}tok/{adaptive_config['max_evidence_chunks']}ch",
        })

        # ---- TIER 0: EVIDENCE SUFFICIENCY GATE ----
        # If evidence is too weak or too sparse, abstain immediately (0 LLM calls).
        # Rationale: no grounded answer is possible without sufficient evidence.
        # Self-assessed complexity determines the threshold:
        #   "complex" questions use a LOWER threshold (0.30) because their evidence
        #   is naturally sparser — aborting too early would reject answerable questions.
        #   "simple" questions use the stricter threshold (0.40).
        if state.retained_evidence:
            mean_relevance = sum(e.relevance_score for e in state.retained_evidence) / len(state.retained_evidence)
        else:
            mean_relevance = 0.0

        # Select threshold based on self-assessed complexity
        if _inferred_complexity == "complex":
            _tier0_threshold = EARLY_STOPPING_CONFIG["evidence_min_mean_score_hard"]
        else:
            _tier0_threshold = EARLY_STOPPING_CONFIG["evidence_min_mean_score"]

        evidence_insufficient = (
            len(state.retained_evidence) < adaptive_config.get("min_evidence_chunks", 1)
            or mean_relevance < _tier0_threshold
        )

        if evidence_insufficient:
            early_stop_tier = 0
            state.can_answer = False
            state.status = "abstained"
            state.final_answer = (
                f"INSUFFICIENT_EVIDENCE: Cannot provide a grounded answer. "
                f"Retrieved evidence is insufficient (mean relevance={mean_relevance:.3f}, "
                f"retained chunks={len(state.retained_evidence)})."
            )
            state.steps_log.append({
                "step": 3, "agent": "EARLY_STOP_TIER0",
                "action": f"abstain: mean_relevance={mean_relevance:.3f}, chunks={len(state.retained_evidence)}",
                "status": "ABSTAINED",
                "latency_ms": 0,
                "ffn_score": 0,
                "details": f"threshold={_tier0_threshold} (inferred={_inferred_complexity}, mean_sim={_mean_retrieval_sim:.3f}), 0 LLM calls saved",
            })
            # Skip ALL agents — jump to return
            raise _EarlyStopSignal()

        # ---- STEP 3: RUN SELECTED AGENTS (pre-SQL) ----
        pre_sql_agents = [a for a in state.selected_agents
                          if MAS_AGENT_ORDER.index(a) < MAS_AGENT_ORDER.index("SQL_DEVELOPER")]
        for agent_name in pre_sql_agents:
            state = run_dynamic_agent(state, agent_name)
            if not state.can_answer:
                break

        # ---- STEP 4: RUN SQL_DEVELOPER ----
        if state.can_answer and "SQL_DEVELOPER" in state.selected_agents:
            state = run_dynamic_agent(state, "SQL_DEVELOPER")

        # ---- TIER 1: SQL FAILED after retries → abstain ----
        if state.can_answer and state.sql_execution_status == "failed":
            early_stop_tier = 1
            state.status = "abstained"
            state.final_answer = (
                f"INSUFFICIENT_EVIDENCE: SQL query failed after {state.sql_attempts} attempts. "
                f"Cannot provide a grounded answer without valid query results."
            )
            state.steps_log.append({
                "step": len(state.steps_log) + 1, "agent": "EARLY_STOP_TIER1",
                "action": f"abstain: SQL failed after {state.sql_attempts} attempts",
                "status": "ABSTAINED",
                "latency_ms": 0,
                "ffn_score": 0,
                "details": "Skipped post-SQL agents (no valid data to interpret)",
            })
            raise _EarlyStopSignal()

        # ---- STEP 5: TEMPORAL RE-SCORING (after SQL) ----
        if state.can_answer:
            state.temporal_scores = temporal_rescore(
                state.initial_scores, state.sql_execution_status, state.sql_result_rows
            )

            # ---- TIER 2: SIMPLE SCALAR BYPASS ----
            # If SQL result is a single scalar (≤1 row, ≤2 cols), skip QUANTITATIVE_ANALYST.
            # A scalar value doesn't need numerical interpretation — it IS the answer.
            # QUALITY_AUDITOR still runs to validate groundedness.
            is_scalar = _is_scalar_result(state.sql_results)

            # Re-select post-SQL agents based on temporal scores
            post_sql_candidates = [a for a in MAS_AGENT_ORDER
                                   if MAS_AGENT_ORDER.index(a) > MAS_AGENT_ORDER.index("SQL_DEVELOPER")]
            post_sql_agents = [a for a in post_sql_candidates
                               if a in select_agents(state.temporal_scores)]

            # Update agent_modes for post-SQL agents based on temporal scores.
            # Without this, an agent resurrected by temporal boost would still have
            # mode="skipped" from initial scoring.
            _temporal_modes = get_agent_modes(state.temporal_scores)
            for _post_agent in post_sql_candidates:
                state.agent_modes[_post_agent] = _temporal_modes[_post_agent]

            # Apply Tier 2: remove QUANTITATIVE_ANALYST for scalar results
            if is_scalar and "QUANTITATIVE_ANALYST" in post_sql_agents:
                early_stop_tier = 2
                post_sql_agents.remove("QUANTITATIVE_ANALYST")
                state.steps_log.append({
                    "step": len(state.steps_log) + 1, "agent": "EARLY_STOP_TIER2",
                    "action": "skip QUANTITATIVE_ANALYST: scalar result (≤1 row, ≤2 cols)",
                    "status": "SCALAR_BYPASS",
                    "latency_ms": 0,
                    "ffn_score": state.temporal_scores.get("QUANTITATIVE_ANALYST", 0),
                    "details": f"SQL result is trivially interpretable, 1 LLM call saved",
                })

            # Run post-SQL agents
            for agent_name in post_sql_agents:
                state = run_dynamic_agent(state, agent_name)
                if not state.can_answer:
                    break

        # ---- STEP 6: FALLBACK if no QUALITY_AUDITOR produced answer ----
        if state.final_answer is None and state.can_answer:
            if state.agent_outputs:
                last_agent = list(state.agent_outputs.keys())[-1]
                state.final_answer = state.agent_outputs[last_agent]
                state.status = "success"
            else:
                state.final_answer = "No agents produced output."
                state.status = "failed"

    except _EarlyStopSignal:
        pass  # Early stop tiers use this to jump to return cleanly
    except Exception as e:
        state.status = "failed"
        state.error = str(e)
        # Leave state.final_answer as None — error details are in state.error.
        # Returning None correctly signals "no answer produced" to the evaluator.

    total_latency = time.time() - start_time

    return {
        "answer": state.final_answer,
        "plan": state.agent_outputs.get("DOMAIN_EXPERT", ""),
        "sql_generated": state.sql_generated,
        "sql_results": state.sql_results,
        "sql_execution_status": state.sql_execution_status,
        "retrieved_chunks": state.raw_chunks,
        "agent_outputs": state.agent_outputs,
        "steps_log": state.steps_log,
        "prompt_tokens": state.total_prompt_tokens,
        "completion_tokens": state.total_completion_tokens,
        "total_tokens": state.total_prompt_tokens + state.total_completion_tokens,
        "latency_seconds": round(total_latency, 2),
        "llm_calls": state.llm_calls,
        "sql_attempts": state.sql_attempts,
        # FFN-specific metadata
        "ffn_scores": state.initial_scores,
        "temporal_scores": state.temporal_scores,
        "selected_agents": state.selected_agents,
        "skipped_agents": state.skipped_agents,
        "agent_modes": state.agent_modes,  # v2: full/compressed/skipped per agent
        "num_active_agents": len(state.agent_outputs),
        "agents_used": list(state.agent_outputs.keys()),
        # Evidence filtering
        "initial_evidence_count": len(state.raw_chunks),
        "retained_evidence_count": len(state.retained_evidence),
        "evidence_retention_rate": len(state.retained_evidence) / max(1, len(state.raw_chunks)),
        # Adaptive budget metadata
        "adaptive_token_budget": adaptive_config["max_context_tokens"],
        "adaptive_max_chunks": adaptive_config["max_evidence_chunks"],
        # Early stopping metadata
        "early_stop_tier": early_stop_tier,
        # Status
        "success": state.status in ("success", "abstained"),
        "status": state.status,
        "error": state.error,
        "is_mock": False,
    }


print("✓ run_dynamic_filtered_mas_rag() defined")
print(f"  FFN gating: full>={FFN_THRESHOLD}, compressed>={FFN_COMPRESSED_THRESHOLD}, mandatory=[DOMAIN_EXPERT, DATA_ENGINEER, SQL_DEVELOPER, QUALITY_AUDITOR]")
print(f"  SQL retries: MAX_SQL_RETRIES={MAX_SQL_RETRIES} (attempt 3 = escalated context)")
print(f"  Self-assessed complexity: mean_retrieval_sim < {COMPLEXITY_THRESHOLD} → 'complex' (full budget)")
print(f"  Adaptive evidence (v3): sim-based budget via SIMILARITY_BUDGET_MAP (decoupled from agent count)")
print(f"  Early stopping:")
print(f"    Tier 0: simple<{EARLY_STOPPING_CONFIG['evidence_min_mean_score']}, complex<{EARLY_STOPPING_CONFIG['evidence_min_mean_score_hard']} → abstain")
print(f"    Tier 1: SQL FAILED → abstain (skip post-SQL agents)")
print(f"    Tier 2: scalar result (≤{EARLY_STOPPING_CONFIG['scalar_max_rows']}row, ≤{EARLY_STOPPING_CONFIG['scalar_max_cols']}cols) → skip QUANTITATIVE_ANALYST")

✓ run_dynamic_filtered_mas_rag() defined
  FFN gating: threshold=0.5, mandatory=[SQL_DEVELOPER, QUALITY_AUDITOR]
  Temporal: re-score after SQL execution
  Evidence filter: dedup + relevance + token budget


In [0]:
# ============================================================
# RUN DYNAMIC_FILTERED_MAS_RAG EXPERIMENT
# ============================================================

# Respect orchestrator filter
if 'QUESTION_FILTER' not in dir():
    QUESTION_FILTER = None
run_questions = [q for q in EVALUATION_QUESTIONS if QUESTION_FILTER is None or q["id"] in QUESTION_FILTER]

print(f"\n{'='*70}")
print(f"DYNAMIC_FILTERED_MAS_RAG: {len(run_questions)} questions | FFN-gated agents | {MODEL_ENDPOINT}")
print(f"{'='*70}")
print(f"  FFN threshold: {FFN_THRESHOLD} | Mandatory: SQL_DEVELOPER, QUALITY_AUDITOR, DOMAIN_EXPERT, DATA_ENGINEER")
print(f"  Evidence filtering: ON (threshold={FILTER_CONFIG['relevance_threshold']})")
print(f"  Temporal re-scoring: ON (after SQL execution)")
print()

all_results = []

for q in run_questions:
    run_id = str(uuid.uuid4())

    try:
        # Execute
        result = run_dynamic_filtered_mas_rag(q)

        # Compute evaluation metrics (same evaluator as all architectures)
        metrics = compute_all_metrics(
            run={"answer": result["answer"], "sql_generated": result.get("sql_generated"),
                 "sql_results": result.get("sql_results"), "success": result["success"],
                 "error": result.get("error")},
            question={"expected_answer": q["expected_answer"], "tables_needed": q.get("tables_needed", []),
                      "answerability_label": q.get("answerability_label", "answerable"),
                      "expected_claims": q.get("expected_claims", []),
                      "question": q["question"]},
            mode=ARCHITECTURE
        )
    except Exception as _loop_err:
        print(f"  {q['id']}: \u2717 FAILED \u2014 {type(_loop_err).__name__}: {_loop_err}")
        all_results.append({
            "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
            "question_type": q["question_type"], "difficulty": q["difficulty"],
            "expected_answer": q["expected_answer"],
            "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
            "semantic_delivery": "retrieved", "profile_name": "Dynamic Filtered MAS",
            "model": MODEL_ENDPOINT,
            "plan": None, "sql_generated": None, "sql_results": None,
            "sql_execution_status": "failed",
            "generated_answer": None,
            "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
            "number_of_model_calls": 0, "llm_calls": 0,
            "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
            "sql_required": bool(q.get("tables_needed")),
            "sql_retry_count": 0,
            "num_active_agents": 0, "number_of_active_agents": 0,
            "number_of_dropped_agents": 0, "number_of_active_edges": 0,
            "latency_seconds": 0.0, "sql_attempts": 0,
            "workflow_status": "failed",
            "success": False, "error": f"{type(_loop_err).__name__}: {_loop_err}",
        })
        continue

    # Compact output — v2: show agent modes (full/compressed/skipped)
    agents_info = f"agents={result['num_active_agents']}/{len(MAS_AGENT_ORDER)}"
    filter_info = f"ev={result['initial_evidence_count']}→{result['retained_evidence_count']}"
    _modes = result.get('agent_modes', {})
    _compressed = [a for a, m in _modes.items() if m == 'compressed']
    _skipped = [a for a, m in _modes.items() if m == 'skipped']
    mode_info = ""
    if _compressed:
        mode_info += f" comp={_compressed}"
    if _skipped:
        mode_info += f" skip={_skipped}"
    print(f"  {q['id']}: {result['latency_seconds']:.1f}s | {result['total_tokens']} tok | "
          f"SQL:{result['sql_execution_status']} | {agents_info} | {filter_info}{mode_info}")

    # --- Derive DYNAMIC-specific KPIs ---
    _selected = result.get("selected_agents", [])
    _active = result.get("num_active_agents", 0)
    _dropped = len(_selected) - _active if _selected else 0
    # Active edges = communication links actually used (from COMMUNICATION_ROUTING)
    _active_edges = 0
    _agents_used = result.get("agents_used", [])
    for _ag in _agents_used:
        _active_edges += len(COMMUNICATION_ROUTING.get(_ag, []))
    # Early stopping: early_stop_tier is None (completed) or 0/1/2 (stopped early)
    _es_tier = result.get("early_stop_tier")
    _early_stop = _es_tier is not None
    _early_stage = None
    if _es_tier == 0: _early_stage = "tier_0_evidence"
    elif _es_tier == 1: _early_stage = "tier_1_sql"
    elif _es_tier == 2: _early_stage = "tier_2_scalar"
    _term_reason = _early_stage or "completed"

    # Store full result
    all_results.append({
        "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
        "question_type": q["question_type"], "difficulty": q["difficulty"],
        "expected_answer": q["expected_answer"],
        "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
        "semantic_delivery": "retrieved", "profile_name": "Dynamic Filtered MAS",
        "model": MODEL_ENDPOINT,
        # Core outputs
        "plan": result.get("plan"),
        "sql_generated": result.get("sql_generated"),
        "sql_results": str(result.get("sql_results", ""))[:500],
        "sql_execution_status": result.get("sql_execution_status"),
        "generated_answer": result.get("answer"),
        # Token breakdown
        "prompt_tokens": result.get("prompt_tokens", 0),
        "completion_tokens": result.get("completion_tokens", 0),
        "total_tokens": result["total_tokens"],
        "number_of_model_calls": result["llm_calls"],
        "llm_calls": result["llm_calls"],
        "tokens_by_agent": str(result.get("tokens_by_agent", {})),
        "communication_tokens": result.get("communication_tokens", 0),
        "retrieved_context_tokens": result.get("retrieved_context_tokens", 0),
        "retained_context_tokens": result.get("retained_context_tokens", 0),
        # Thesis KPIs (from question metadata)
        "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
        "sql_required": bool(q.get("tables_needed")),
        "sql_retry_count": result.get("sql_attempts", 1) - 1,
        # FFN metadata
        "ffn_scores": str(result.get("ffn_scores", {})),
        "temporal_scores": str(result.get("temporal_scores", {})),
        "selected_agents": str(_selected),
        "skipped_agents": str(result.get("skipped_agents", [])),
        "num_active_agents": _active,
        "number_of_active_agents": _active,
        "number_of_dropped_agents": _dropped,
        "number_of_active_edges": _active_edges,
        "agents_used": str(_agents_used),
        # Evidence filtering
        "retrieved_chunks": result.get("initial_evidence_count", 0),
        "retained_chunks": result.get("retained_evidence_count", 0),
        "initial_evidence_count": result.get("initial_evidence_count", 0),
        "retained_evidence_count": result.get("retained_evidence_count", 0),
        "evidence_retention_rate": result.get("evidence_retention_rate", 0),
        # Early stopping & termination
        "early_stopping_triggered": _early_stop,
        "early_stopping_stage": _early_stage,
        "termination_reason": _term_reason,
        # Execution
        "latency_seconds": result["latency_seconds"],
        "sql_attempts": result.get("sql_attempts", 0),
        "workflow_status": result["status"],
        "success": result["success"], "error": result.get("error"),
        # Evaluation metrics (includes all thesis KPIs from compute_all_metrics)
        **metrics,
    })

results_df = pd.DataFrame(all_results)

# ---- SUMMARY ----
print(f"\n{'='*70}")
print(f"DYNAMIC_FILTERED_MAS_RAG COMPLETE: {len(results_df)} runs")
print(f"{'='*70}")
if len(results_df) > 0:
    print(f"  SQL success:        {(results_df['sql_execution_status']=='success').sum()}/{len(results_df)}")
    print(f"  Avg latency:        {results_df['latency_seconds'].mean():.2f}s")
    print(f"  Total tokens:       {results_df['total_tokens'].sum():,}")
    print(f"  Avg tokens/q:       {results_df['total_tokens'].mean():.0f}")
    print(f"  Avg agents/q:       {results_df['num_active_agents'].mean():.1f}/{len(MAS_AGENT_ORDER)}")
    print(f"  Evidence filtered:  avg {results_df['initial_evidence_count'].mean():.1f}→{results_df['retained_evidence_count'].mean():.1f} chunks")
    if 'answer_f1' in results_df.columns:
        print(f"  Mean F1:            {results_df['answer_f1'].mean():.3f}")
        print(f"  Mean Precision:     {results_df['answer_precision'].mean():.3f}")
        print(f"  Mean Recall:        {results_df['answer_recall'].mean():.3f}")

    # Agent selection distribution
    print(f"\n  Agent activation frequency:")
    for agent in MAS_AGENT_ORDER:
        count = results_df['agents_used'].apply(lambda x: agent in x).sum()
        print(f"    {agent}: {count}/{len(results_df)} questions")


DYNAMIC_FILTERED_MAS_RAG: 25 questions | FFN-gated agents | databricks-meta-llama-3-1-8b-instruct
  FFN threshold: 0.5 | Mandatory: SQL_DEVELOPER, QUALITY_AUDITOR
  Evidence filtering: ON (threshold=0.35)
  Temporal re-scoring: ON (after SQL execution)

  Q1: 5.7s | 13026 tok | SQL:success | agents=3/5 | ev=5→5 skip=['DATA_ENGINEER']
  Q2: 7.8s | 19092 tok | SQL:success | agents=4/5 | ev=5→5 skip=['DATA_ENGINEER']
  Q3: 10.2s | 24742 tok | SQL:success | agents=5/5 | ev=5→5 skip=['QUANTITATIVE_ANALYST']
  Q4: 12.5s | 44934 tok | SQL:failed | agents=4/5 | ev=5→5 skip=['DATA_ENGINEER']
  Q5: 8.2s | 18942 tok | SQL:success | agents=4/5 | ev=5→5 
  Q6: 8.6s | 24096 tok | SQL:empty | agents=5/5 | ev=5→5 
  Q7: 14.3s | 55413 tok | SQL:failed | agents=5/5 | ev=5→5 
  Q8: 12.1s | 48091 tok | SQL:failed | agents=5/5 | ev=5→5 
  Q9: 14.7s | 56147 tok | SQL:failed | agents=5/5 | ev=5→5 
  Q10: 12.8s | 19209 tok | SQL:empty | agents=4/5 | ev=5→5 skip=['QUANTITATIVE_ANALYST']
  Q11: 3.2s | 11769 to

In [0]:
# ============================================================
# CAPTURE DYNAMIC_FILTERED_MAS_RAG RESULTS + PERSIST TO DELTA
# ============================================================
dynamic_filtered_results_df = None

if "DYNAMIC_FILTERED_MAS_RAG" in ARCHITECTURES_TO_RUN and 'results_df' in dir() and len(results_df) > 0:
    dynamic_filtered_results_df = results_df.copy()
    all_experiment_results.extend(results_df.to_dict('records'))
    print(f"✓ DYNAMIC_FILTERED_MAS_RAG captured: {len(dynamic_filtered_results_df)} runs")
    # Show token efficiency vs MAS_RAG
    if mas_rag_results_df is not None and 'total_tokens' in dynamic_filtered_results_df.columns:
        mas_avg = mas_rag_results_df['total_tokens'].mean()
        dyn_avg = dynamic_filtered_results_df['total_tokens'].mean()
        reduction = (1 - dyn_avg / mas_avg) * 100 if mas_avg > 0 else 0
        print(f"  Token efficiency vs MAS_RAG: {reduction:+.1f}% ({int(dyn_avg)} vs {int(mas_avg)} avg tokens/question)")
else:
    print("⚠ DYNAMIC_FILTERED_MAS_RAG not run or no results")

_print_cumulative_comparison()
print(f"\n✓ All architectures complete: {len(all_experiment_results)} total runs")

# --- Persist to Delta ---
try:
    if dynamic_filtered_results_df is not None:
        _sdf = spark.createDataFrame(dynamic_filtered_results_df.astype(str))
        _sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(_RESULTS_TABLE)
        print(f"✓ DYNAMIC_FILTERED_MAS_RAG appended to {_RESULTS_TABLE}")
except Exception as _e:
    print(f"⚠ Delta persist failed: {_e}")

import gc; gc.collect()